<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/text_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ftfy contractions -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00


In [2]:
import re
import string
import unicodedata
import ftfy
import contractions

# ============================================
# Read the file into your notebook
# ============================================
with open('text_sample.txt', 'r', encoding='utf-8', errors='replace') as f:
    text = f.read()

print("--- ORIGINAL TEXT (first 500 chars) ---")
print(text[:500])

# ============================================
# Step 1: Fix encoding issues (do this FIRST, before other cleaning)
# ============================================
# ftfy fixes "mojibake" — garbled text from encoding mismatches (e.g. â€™ instead of ')
text = ftfy.fix_text(text)

# unicodedata normalizes characters (e.g. accented letters, smart quotes -> normal form)
text = unicodedata.normalize('NFKC', text)

# ============================================
# Step 2: Remove HTML tags
# ============================================
text = re.sub(r'<[^>]+>', ' ', text)

# ============================================
# Step 3: Expand contractions (do this before removing punctuation,
# since contractions rely on apostrophes)
# ============================================
text = contractions.fix(text)

# ============================================
# Step 4: Handle emojis — remove them
# ============================================
def remove_emojis(input_text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U0001F900-\U0001F9FF"  # supplemental symbols
        "\U00002600-\U000026FF"  # misc symbols
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', input_text)

text = remove_emojis(text)

# ============================================
# Step 5: Remove random symbols like @@, ##, and similar noise
# ============================================
# Remove repeated special characters (e.g. @@, ##, **, ~~)
text = re.sub(r'[@#\$%\^&\*~`]{1,}', ' ', text)

# ============================================
# Step 6: Standardize inconsistent values (example: price formats)
# ============================================
# Standardize things like "$ 10", "USD10", "10 dollars" -> "$10"
text = re.sub(r'\bUSD\s?(\d+(\.\d{1,2})?)', r'$\1', text, flags=re.IGNORECASE)
text = re.sub(r'\$\s+(\d)', r'$\1', text)                     # "$ 10" -> "$10"
text = re.sub(r'(\d+(\.\d{1,2})?)\s?dollars?', r'$\1', text, flags=re.IGNORECASE)

# ============================================
# Step 7: Convert to lowercase
# ============================================
text = text.lower()

# ============================================
# Step 8: Remove unnecessary punctuation
# ============================================
# Keep $ and . if you want prices/decimals to survive — adjust as needed.
# This version removes ALL standard punctuation:
punctuation_to_remove = string.punctuation
text = text.translate(str.maketrans('', '', punctuation_to_remove))

# ============================================
# Step 9: Remove extra spaces and newlines
# ============================================
# Do this LAST, since earlier steps (removing tags/symbols/punctuation)
# leave behind extra whitespace gaps
text = re.sub(r'\s+', ' ', text).strip()

# ============================================
# Final result
# ============================================
print("\n--- CLEANED TEXT (first 500 chars) ---")
print(text[:500])

# ============================================
# Save the cleaned text to a new file
# ============================================
with open('cleaned_text.txt', 'w', encoding='utf-8') as f:
    f.write(text)

# Download it from Colab
from google.colab import files
files.download('cleaned_text.txt')

--- ORIGINAL TEXT (first 500 chars) ---
Thís ís à próblemátic téxt fíle!! It contains    **extra spaces** ,,,,, special characters!!!💥🔥🚀
Some words are misspelleddd,   and encoding  issues   liké thís cäusé problëms.    
        
Prices are inconsistent:  $29.99, 29.99 USD, 29,99$.

Emails & phone numbers may be embedded: contact@domain.com, (123)-456-7890.

Repeated punctuations!!!!! should be removed, along with **random symbols** like @@,##.

stopwords like "the", "is", and "a" appear often.

HTML tags might be present: <div>This i

--- CLEANED TEXT (first 500 chars) ---
thís ís à próblemátic téxt fíle it contains extra spaces special characters some words are misspelleddd and encoding issues liké thís cäusé problëms prices are inconsistent 2999 2999 usd 2999 emails phone numbers may be embedded contact domaincom 1234567890 repeated punctuations should be removed along with random symbols like stopwords like the is and a appear often html tags might be present this is inside a div a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>